# SIH26142 — Satellite Super-Resolution — FAST Colab Fine-Tuning

**This notebook replaces `Finetune_On_Colab.ipynb`.** Same pipeline, same code — the
difference is what data path runs by default and how the training loop is configured.

### Why the old notebook took hours

Run this top to bottom on a Colab **T4 GPU** runtime (Runtime → Change runtime type → T4 GPU),
and the default path here should finish in **roughly 15-25 minutes**, not hours. The old
notebook's "3b. WorldStrat raw GeoTIFFs" cell downloaded the **entire worldwide WorldStrat
archive** — confirmed directly against WorldStrat's own Zenodo listing: `hr_dataset.tar.gz`
is **40.8 GB** and `lr_dataset_l2a.tar.gz` is **25.5 GB**, i.e. **~66 GB** for two files, covering
every AOI on Earth, not just the handful you need for one Indian land-use story. On Colab's
often-throttled bandwidth to Zenodo, that download alone can take 1-3+ hours, before you've
even tiled it (tiling *every* global AOI, not just 30-50 relevant ones, multiplies your patch
count and your first-run preprocessing time far beyond what a 3-week idea-stage prototype
needs) or trained on it (more patches x more epochs = more wall-clock GPU time).

**This notebook's default path never touches that download.** It uses `opensr-test` —
already-aligned, pre-packaged, small pip-installable datasets — combined across 4 of its
5 bundled sets (`spot` + `naip` + `spain_crops` + `spain_urban` = 119 real scenes, all x4
scale) instead of just `spot` alone (9 scenes). The full WorldStrat raw path is still here
if you genuinely need more volume later — it's moved to a clearly-marked **Optional /
Advanced** section at the end, gated behind a flag you must set to `True` yourself, with a
disk-space check before it downloads anything.

### What else changed (see `src/train.py` / `src/data_pipeline.py` for full detail)
- Mixed precision (AMP) — ~1.5-2x fewer seconds per step on a T4.
- `freeze_backbone=True` by default — trains only the last 4 RRDB blocks + the upsampling
  head (~standard transfer-learning practice for a small dataset), not all 16.7M params.
- Checkpointing writes `checkpoints/best.pth` + `checkpoints/last.pth` only — not a new
  ~64MB file every single epoch (the old notebook's 30-epoch default wrote ~1.9GB to Drive).
- `build_patches_from_opensr_test_multi()` fixes a real bug in the original single-dataset
  helper: calling it twice at the same `out_dir` silently overwrote patches from the first
  call (no crash, no warning) — the multi-dataset version indexes patches correctly.
- Epoch 1 prints a wall-clock estimate for the whole run **before** you've sat through it.

**Before running:** upload the updated `sih26142-superres` project zip to this Colab
session's file browser, or mount Drive and place it there (same as before).


## 1. Environment setup (fast path — no GDAL/rasterio needed by default)

In [ ]:
# Colab ships torch pre-installed with CUDA — do NOT reinstall it, that
# breaks the GPU build. We only need the few extra packages this project needs.
# NOTE: rasterio/GDAL/sentinelhub are deliberately NOT installed here — they're
# only needed for the Optional/Advanced WorldStrat-raw and Copernicus sections
# at the end of this notebook, not for the default fast path.
!pip install -q basicsr --no-deps
!pip install -q addict future yapf tqdm scikit-image opensr-test

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (wrong runtime! Runtime > Change runtime type > T4 GPU)")
assert torch.cuda.is_available(), "Stop here and switch to a T4 GPU runtime before continuing — everything below assumes a GPU."


## 2. Mount Drive (recommended — checkpoints/data survive a disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/sih26142-superres"


In [ ]:
# Unzip the project (upload the project zip to Colab's file browser first,
# or point this at wherever you placed it in Drive)
import shutil, zipfile, os

ZIP_PATH = "/content/SIH26142_WorkingPrototype.zip"  # <- edit if you placed it elsewhere

if not os.path.exists(PROJECT_DIR):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("/content/drive/MyDrive/")
    print("Extracted to", PROJECT_DIR)
else:
    print("Project already exists at", PROJECT_DIR, "- skipping re-extract")

import sys
sys.path.insert(0, f"{PROJECT_DIR}/src")
os.chdir(PROJECT_DIR)


## 3. Get the pretrained Real-ESRGAN weights

67MB, official release — same file the local prototype validated against. Small, fast, unchanged from before.

In [ ]:
!mkdir -p experiments/pretrained_models
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth \
    -P experiments/pretrained_models
!ls -la experiments/pretrained_models/


## 4. Build the training set — combined `opensr-test` datasets

Real, ALREADY spatially-aligned Sentinel-2 <-> SPOT/NAIP/aerial pairs — ESA-funded, IEEE GRSL
2024, zero GeoTIFF/CRS/cloud-masking work needed. This combines 4 of `opensr-test`'s 5 bundled
datasets (all x4 scale — `venus` is x2 and deliberately excluded so LR/HR size ratios stay
consistent): `spot` (9 scenes, the same WorldStrat SPOT/Sentinel-2 source this project's
research brief already cites) + `naip` (62) + `spain_crops` (28) + `spain_urban` (20) = **119
real aligned scenes**, vs. 9 if you only use `spot` alone. Total download here is tens of MB,
not tens of GB — this whole cell should take well under a minute.

If your chosen Indian AOI story is specifically agriculture, urban fringe, or coastline, you
can drop `spain_urban`/`spain_crops`/`naip` from the tuple below and lean on whichever land-use
type matches your pitch best — more scenes is usually still better than fewer, though.

In [ ]:
import sys
sys.path.insert(0, "src")
from data_pipeline import build_patches_from_opensr_test_multi, split_train_val

n_written = build_patches_from_opensr_test_multi(
    names=("spot", "naip", "spain_crops", "spain_urban"),
    out_dir="data/processed",
)
split_train_val("data/processed", "data/processed_val", val_fraction=0.15, seed=42)


## 5. Fine-tune (fast: mixed precision + partial-freeze)

With 119 source scenes x augmentation, a T4 GPU, AMP, and only the last 4 RRDB blocks +
upsampling head trainable, this should take on the order of **single-digit minutes** for the
default 15 epochs — the training cell prints a wall-clock estimate after epoch 1, so you'll
know immediately if something is off (e.g. you're accidentally on a CPU runtime).

Set `freeze_backbone=False` only if you've confirmed the fast path is genuinely too limited
for your pitch AND you have real time budget left — it fine-tunes all 16.7M parameters
instead of the last few blocks, which is slower per step and needs more data to avoid
overfitting.

In [ ]:
import sys
sys.path.insert(0, "src")
from train import train

model = train(
    lr_dir="data/processed/lr",
    hr_dir="data/processed/hr",
    pretrained_weights="experiments/pretrained_models/RealESRGAN_x4plus.pth",
    val_lr_dir="data/processed_val/lr",   # tracks best.pth by validation loss, not train loss
    val_hr_dir="data/processed_val/hr",
    epochs=15,
    batch_size=16,          # T4 has 16GB VRAM — raise if you have headroom, lower if you hit OOM
    lr=1e-4,
    patch_size=128,
    scale=4,
    checkpoint_dir="checkpoints",
    freeze_backbone=True,   # see markdown above
    device="cuda",
)
# train() saves checkpoints/best.pth (lowest validation loss) and checkpoints/last.pth
# (most recent epoch, for disconnect recovery) — not one file per epoch.


## 6. Evaluate against a held-out set

Compares your fine-tuned model against both the pretrained-but-not-fine-tuned baseline and plain bicubic — the exact table for your results slide.

In [ ]:
import sys
sys.path.insert(0, "src")
from dataset import SentinelSRDataset
from model import load_realesrgan
from evaluate import evaluate_pair, bicubic_upsample
import torch
import numpy as np

val_dataset = SentinelSRDataset("data/processed_val/lr", "data/processed_val/hr",
                                  patch_size=128, scale=4, augment=False)

finetuned = load_realesrgan("checkpoints/best.pth", scale=4, device="cuda")
finetuned.eval()
generic = load_realesrgan("experiments/pretrained_models/RealESRGAN_x4plus.pth", scale=4, device="cuda")
generic.eval()

rows = {"finetuned": [], "generic_pretrained": [], "bicubic": []}
with torch.no_grad():
    for i in range(len(val_dataset)):
        lr_t, hr_t = val_dataset[i]
        lr_np = lr_t.permute(1, 2, 0).numpy()
        hr_np = hr_t.permute(1, 2, 0).numpy()
        lr_batch = lr_t.unsqueeze(0).cuda()

        sr_ft = finetuned(lr_batch).squeeze(0).permute(1, 2, 0).cpu().clamp(0, 1).numpy()
        sr_gen = generic(lr_batch).squeeze(0).permute(1, 2, 0).cpu().clamp(0, 1).numpy()
        bic = bicubic_upsample(lr_np, scale=4)

        rows["finetuned"].append(evaluate_pair(sr_ft, hr_np, resolution_ratio=2.5))
        rows["generic_pretrained"].append(evaluate_pair(sr_gen, hr_np, resolution_ratio=2.5))
        rows["bicubic"].append(evaluate_pair(bic, hr_np, resolution_ratio=2.5))

print(f"{'method':22s} {'PSNR':>8s} {'SSIM':>8s} {'SAM':>8s} {'ERGAS':>8s}")
for name, results in rows.items():
    avg = {k: np.mean([r[k] for r in results]) for k in results[0]}
    print(f"{name:22s} {avg['PSNR']:8.2f} {avg['SSIM']:8.4f} {avg['SAM_deg']:8.3f} {avg['ERGAS']:8.2f}")
print("\nThis is the table that goes on your results slide — finetuned should beat both")
print("generic_pretrained AND bicubic on PSNR/SSIM (higher better) and SAM/ERGAS (lower better).")


## 7. Visual comparison grid (for your PPT)

Saves a figure with LR input | bicubic | your model | ground truth, side by side — the single most persuasive slide for non-technical judges.

In [ ]:
import matplotlib.pyplot as plt

n_examples = min(4, len(val_dataset))
fig, axes = plt.subplots(n_examples, 4, figsize=(16, 4 * n_examples))
if n_examples == 1:
    axes = axes[None, :]

for i in range(n_examples):
    lr_t, hr_t = val_dataset[i]
    lr_np = lr_t.permute(1, 2, 0).numpy()
    hr_np = hr_t.permute(1, 2, 0).numpy()
    lr_batch = lr_t.unsqueeze(0).cuda()

    with torch.no_grad():
        sr = finetuned(lr_batch).squeeze(0).permute(1, 2, 0).cpu().clamp(0, 1).numpy()
    bicubic = bicubic_upsample(lr_np, scale=4)

    for ax, img, title in zip(axes[i], [lr_np, bicubic, sr, hr_np],
                                ["LR input (simulated 10m)", "Bicubic", "Fine-tuned model", "Ground truth"]):
        ax.imshow(img)
        ax.set_title(title, fontsize=11)
        ax.axis("off")

plt.tight_layout()
plt.savefig("checkpoints/comparison_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to checkpoints/comparison_grid.png — drop this straight into the Technical Approach or Impact slide.")


## 8. Export the checkpoint for the demo

Copy your best checkpoint to `checkpoints/best.pth` — the Streamlit demo (`demo/app.py`) auto-detects this path and switches from the "generic pretrained" badge to "fine-tuned checkpoint" automatically.

In [ ]:
from pathlib import Path

best_path = Path("checkpoints/best.pth")
assert best_path.exists(), (
    "checkpoints/best.pth wasn't created — check the training cell above ran "
    "without error and that data/processed_val actually has patches in it."
)
print(f"{best_path} is ready ({best_path.stat().st_size / 1e6:.1f} MB).")
print("Download this whole PROJECT_DIR folder from Drive, or just")
print("checkpoints/best.pth, and drop it into your local repo copy.")
print("Then: streamlit run demo/app.py")


---
## Optional / Advanced sections below

Everything below this line is **not part of the default fast path** and won't run unless you
explicitly opt in. Only come back here if the 119-scene `opensr-test` path above genuinely
isn't enough for your pitch and you have real time/disk budget to spend.

### A. Real India Sentinel-2 tile (Copernicus) — for validation only

Gives you REAL Sentinel-2 input (not synthetically degraded) to show in your demo, independent
of which training-data path you used. Needs a free account at
https://browser.dataspace.copernicus.eu — small download (one tile), safe to run any time.

In [ ]:
# Uncomment and fill in your credentials to use this cell.
# !pip install -q sentinelhub
#
# from sentinelhub import SHConfig
# config = SHConfig()
# config.sh_client_id = ""      # <- fill in from your Copernicus account
# config.sh_client_secret = ""  # <- fill in from your Copernicus account
#
# import sys
# sys.path.insert(0, "src")
# from data_pipeline import download_sentinel2_tile
#
# AOI_BBOX = (77.55, 12.90, 77.65, 13.00)  # example: part of Bengaluru — edit to your AOI
# tile = download_sentinel2_tile(AOI_BBOX, ("2026-01-01", "2026-03-01"), "data/raw/india_sample.npy")
# print("Downloaded tile shape:", tile.shape)


### B. ⚠️ Full WorldStrat raw download — 66GB+, likely 1-3+ hours

**This is almost certainly what made your previous run take hours.** `hr_dataset.tar.gz` is
confirmed **40.8 GB** and `lr_dataset_l2a.tar.gz` is confirmed **25.5 GB** (checked directly
against WorldStrat's own Zenodo listing) — together, ~66GB, covering every AOI on Earth. Colab's
free-tier disk is ~78-107GB total, so this alone can eat most of your session's disk, and
Zenodo's download speed is not always fast, so the download step by itself can run 1-3+ hours
depending on the day.

Only run this if you've confirmed the `opensr-test` combined path (119 scenes) above isn't
enough. If you do, **immediately delete every AOI subfolder except the 30-50 matching your
chosen Indian land-use story** before tiling — tiling all of them multiplies both your disk
use and your training set size (and therefore epoch time) far beyond what a 3-week idea-stage
prototype needs.

The cell below is gated behind `RUN_FULL_WORLDSTRAT_DOWNLOAD = False` and checks free disk
space before doing anything — it will refuse to run if you don't have room.

In [ ]:
RUN_FULL_WORLDSTRAT_DOWNLOAD = False  # <- set True yourself if you really want this

import shutil

if not RUN_FULL_WORLDSTRAT_DOWNLOAD:
    print("Skipped (RUN_FULL_WORLDSTRAT_DOWNLOAD is False). This is almost certainly the right choice.")
else:
    free_gb = shutil.disk_usage("/content").free / 1e9
    NEEDED_GB = 80  # ~66GB for the two archives + room to extract them
    print(f"Free disk: {free_gb:.1f} GB (need at least ~{NEEDED_GB} GB)")
    assert free_gb >= NEEDED_GB, (
        f"Not enough free disk ({free_gb:.1f} GB < {NEEDED_GB} GB) — this download WILL fail "
        "partway through and waste the time already spent. Free up space or skip this section."
    )
    # Pulls directly from WorldStrat's Zenodo record (zenodo.org/record/6810792)
    # into two SEPARATE folders -- this is the real archive layout, not a merged per-AOI folder.
    !mkdir -p /content/worldstrat/hr_dataset /content/worldstrat/lr_dataset_l2a
    !wget -q "https://zenodo.org/record/6810792/files/hr_dataset.tar.gz?download=1" -O /content/hr_dataset.tar.gz
    !wget -q "https://zenodo.org/record/6810792/files/lr_dataset_l2a.tar.gz?download=1" -O /content/lr_dataset_l2a.tar.gz
    !tar -xzf /content/hr_dataset.tar.gz -C /content/worldstrat/hr_dataset
    !tar -xzf /content/lr_dataset_l2a.tar.gz -C /content/worldstrat/lr_dataset_l2a
    print("HR sample:"); import os; print(os.listdir("/content/worldstrat/hr_dataset")[:3])
    print("LR sample:"); print(os.listdir("/content/worldstrat/lr_dataset_l2a")[:3])
    print("\nNOW DELETE every AOI subfolder except the 30-50 matching your chosen land-use")
    print("story before running discover_aoi_pairs/build_patches_from_pairs below.")


In [ ]:
# Only meaningful once RUN_FULL_WORLDSTRAT_DOWNLOAD above actually ran and you've
# trimmed the AOI folders down as instructed.
import sys
sys.path.insert(0, "src")
from data_pipeline import discover_aoi_pairs, build_patches_from_pairs, split_train_val

HR_ROOT = "/content/worldstrat/hr_dataset"
LR_ROOT = "/content/worldstrat/lr_dataset_l2a"

pairs = discover_aoi_pairs(HR_ROOT, LR_ROOT)
print(f"Found {len(pairs)} AOI pairs")

n_written = build_patches_from_pairs(
    pairs, out_dir="data/processed_worldstrat",
    raw_hr_patch=256, scale=4,
    use_synthetic_lr=False,
)
split_train_val("data/processed_worldstrat", "data/processed_worldstrat_val", val_fraction=0.15, seed=42)
